# Notebook 4: Building a Knowledge Graph from Web Content

In this notebook, you'll learn how to:
1. Load **PDFs from URLs** using the built-in `PdfLoader` with `fsspec`
2. Create a **custom DataLoader** for web pages (the package's pattern for extending loaders)
3. Use both with `SimpleKGPipeline` to build knowledge graphs

## The `DataLoader` Pattern

The `neo4j-graphrag` package provides a `DataLoader` base class that you extend to handle different content types. The built-in `PdfLoader` already supports remote URLs via `fsspec`. For HTML web pages, we'll create a custom loader following the same pattern.

## Step 1: Setup

In [ ]:
# beautifulsoup4 and requests are needed for our custom web loader
# aiohttp is needed for fsspec to load remote PDFs via HTTPS
!pip install beautifulsoup4 requests aiohttp -q

In [ ]:
import os
import asyncio
from pathlib import Path
import neo4j
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j!")

In [ ]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

llm = OpenAILLM(
    model_name="gpt-4.1-2025-04-14",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

embedder = OpenAIEmbeddings(model="text-embedding-3-large")
print("LLM and Embedder ready!")

## Step 2: Load a Remote PDF with `PdfLoader`

The built-in `PdfLoader` supports loading PDFs from URLs via `fsspec`. Just pass `fs="https"` and give it a URL instead of a local file path.

In [ ]:
from neo4j_graphrag.experimental.components.pdf_loader import PdfLoader

loader = PdfLoader()

# Load a PDF directly from arXiv — no downloading needed!
pdf_url = "https://arxiv.org/pdf/1706.03762v7"
print(f"Loading PDF from: {pdf_url}")

doc = await loader.run(filepath=pdf_url, fs="https")
print(f"Loaded {len(doc.text)} characters")
print(f"Preview: {doc.text[:300]}...")

## Step 3: Build a KG from the Remote PDF

We can pass the loaded text directly to `SimpleKGPipeline`.

In [ ]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    from_pdf=False,  # We already extracted the text via PdfLoader
    neo4j_database=NEO4J_DATABASE,
)

# Use the first 5000 chars to keep this quick
remote_text = doc.text[:5000]

print(f"Processing remote PDF ({len(remote_text)} chars)...")

result = await kg_builder.run_async(
    text=remote_text,
    document_metadata={"source": "arxiv", "url": pdf_url},
)

print("\nDone!")
print(f"Result: {result}")

In [ ]:
# See what was extracted
records, _, _ = driver.execute_query(
    """
    MATCH (a)-[r]->(b)
    WHERE NOT a:Chunk AND NOT a:Document AND NOT b:Chunk AND NOT b:Document
    RETURN labels(a)[0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           labels(b)[0] AS to_type, b.name AS to_name
    ORDER BY from_name
    LIMIT 20
    """,
    database_=NEO4J_DATABASE,
)

print("=== Extracted from Remote PDF (first 20) ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

## Step 4: Create a Custom Web Page Loader

The `neo4j-graphrag` package provides a `DataLoader` base class for loading different content types. The built-in `PdfLoader` extends it for PDFs. We can follow the same pattern to create a loader for HTML web pages.

This is the package's recommended approach for extending the pipeline to new data sources.

In [ ]:
from typing import Dict, Optional
import requests
from bs4 import BeautifulSoup
from neo4j_graphrag.experimental.components.pdf_loader import DataLoader
from neo4j_graphrag.experimental.components.types import DocumentInfo, PdfDocument


class WebPageLoader(DataLoader):
    """Custom DataLoader that fetches and extracts text from HTML web pages.

    Follows the neo4j-graphrag DataLoader pattern so it integrates
    cleanly with the rest of the pipeline.
    """

    async def run(
        self,
        filepath: str,
        metadata: Optional[Dict[str, str]] = None,
        **kwargs,
    ) -> PdfDocument:
        """Fetch a URL and extract its text content.

        Args:
            filepath: The URL to fetch.
            metadata: Optional metadata to attach to the document.
        """
        headers = {"User-Agent": "KG-Workshop/1.0 (Educational)"}
        response = requests.get(filepath, headers=headers, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove non-content elements
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        # Extract and clean text
        text = soup.get_text(separator="\n", strip=True)
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        clean_text = "\n".join(lines)

        return PdfDocument(
            text=clean_text,
            document_info=DocumentInfo(
                path=filepath,
                metadata=self.get_document_metadata(clean_text, metadata),
            ),
        )


# Test the loader
web_loader = WebPageLoader()
web_doc = await web_loader.run(
    filepath="https://en.wikipedia.org/wiki/Knowledge_graph",
    metadata={"source": "wikipedia"},
)

print(f"Loaded {len(web_doc.text)} characters")
print(f"Document path: {web_doc.document_info.path}")
print(f"\nFirst 500 characters:\n{web_doc.text[:500]}")

## Step 5: Build a KG from a Web Page

Now we use the text from our `WebPageLoader` with the pipeline, just like any other data source.

In [ ]:
# Use the first 5000 characters to keep it quick
web_text = web_doc.text[:5000]

print(f"Processing {len(web_text)} characters from Wikipedia...")

result = await kg_builder.run_async(
    text=web_text,
    document_metadata={"source": "wikipedia", "url": web_doc.document_info.path},
)

print("\nDone!")
print(f"Result: {result}")

In [ ]:
# See what was extracted from the web page
records, _, _ = driver.execute_query(
    """
    MATCH (a)-[r]->(b)
    WHERE NOT a:Chunk AND NOT a:Document AND NOT b:Chunk AND NOT b:Document
    RETURN labels(a)[0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           labels(b)[0] AS to_type, b.name AS to_name
    ORDER BY from_name
    LIMIT 20
    """,
    database_=NEO4J_DATABASE,
)

print("=== Extracted from Web Page (first 20) ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

## Step 6: Explore the Full Graph

If you've run all the notebooks, your graph now contains knowledge from:
- Plain text (Notebook 2)
- Local PDFs (Notebook 3)
- Remote PDFs and web pages (this notebook)

Let's see the full picture!

In [ ]:
# Overall graph statistics
records, _, _ = driver.execute_query(
    "MATCH (n) RETURN count(n) AS nodes",
    database_=NEO4J_DATABASE,
)
node_count = records[0]["nodes"]

records, _, _ = driver.execute_query(
    "MATCH ()-[r]->() RETURN count(r) AS rels",
    database_=NEO4J_DATABASE,
)
rel_count = records[0]["rels"]

print(f"=== Full Graph Statistics ===")
print(f"  Total nodes: {node_count}")
print(f"  Total relationships: {rel_count}")

In [ ]:
# All entity types in the graph
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WHERE NOT n:Chunk AND NOT n:Document
    WITH labels(n) AS types, count(n) AS count
    RETURN types, count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Entity Types ===")
for record in records:
    print(f"  {record['types']}: {record['count']}")

In [ ]:
# All documents (sources) in the graph
records, _, _ = driver.execute_query(
    """
    MATCH (d:Document)
    RETURN d.path AS path, d.source AS source, d.url AS url
    """,
    database_=NEO4J_DATABASE,
)

print("=== Source Documents ===")
for record in records:
    source = record.get('source') or 'unknown'
    url = record.get('url') or ''
    print(f"  [{source}] {record['path']} {url}")

In [ ]:
# All relationship types
records, _, _ = driver.execute_query(
    """
    MATCH ()-[r]->()
    WHERE type(r) <> 'FROM_DOCUMENT' AND type(r) <> 'NEXT_CHUNK' AND type(r) <> 'FROM_CHUNK'
    RETURN type(r) AS rel_type, count(r) AS count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Relationship Types ===")
for record in records:
    print(f"  {record['rel_type']}: {record['count']}")

## Bonus: Process Multiple Web Pages

Here's a pattern for processing multiple URLs in sequence using our custom loader.

In [ ]:
# Example: process multiple URLs (uncomment to run)

# urls = [
#     "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)",
#     "https://en.wikipedia.org/wiki/Large_language_model",
# ]
#
# for url in urls:
#     print(f"\nLoading: {url}")
#     doc = await web_loader.run(filepath=url, metadata={"source": "wikipedia"})
#     text = doc.text[:5000]  # First 5000 chars
#     result = await kg_builder.run_async(
#         text=text,
#         document_metadata={"source": "wikipedia", "url": url},
#     )
#     print(f"  Done! {result}")

## Summary

In this notebook you learned how to:
- Load remote PDFs with the built-in `PdfLoader` and `fs="https"`
- Create a custom `DataLoader` subclass for web pages (following the package's pattern)
- Build knowledge graphs from web content
- Combine multiple data sources into a single graph

## Workshop Recap

Across all four notebooks, you've learned to build knowledge graphs from:

| Source | Loader | Pipeline Setting |
|--------|--------|------------------|
| Plain text | None (direct input) | `from_pdf=False`, pass `text=` |
| Local PDF | Built-in `PdfLoader` | `from_pdf=True` (default), pass `file_path=` |
| Remote PDF | `PdfLoader` with `fs="https"` | Load text first, then `from_pdf=False` |
| Web page | Custom `WebPageLoader` | Load text first, then `from_pdf=False` |

**What's next?** Explore your graph in the [Neo4j Browser](https://console.neo4j.io) — you can visualize nodes and relationships interactively!

In [ ]:
driver.close()